# 🤖 1주차: AI 차트 리더 만들기

> **미션:** AI에게 내 주식 차트를 보여주고 분석을 받는다.

## 오늘의 흐름
1. `pykrx`로 테마 대장주 1년치 데이터 수집
2. `plotly`로 이동평균선 포함 인터랙티브 캔들차트
3. 차트 이미지를 `GPT-4o Vision`에게 전달 → AI 퀀트 의견 획득
4. 결과를 CSV로 저장 (2주차 재료)

---

## Step 0. 환경 세팅

In [ ]:
# 라이브러리 설치 (Colab 런타임당 1회)
!pip install pykrx openai plotly kaleido -q

In [ ]:
import base64
from datetime import datetime, timedelta

import pandas as pd
import plotly.graph_objects as go
from pykrx import stock
from openai import OpenAI

### 🔑 OpenAI API Key 등록

1. Colab 왼쪽 **🔑 아이콘** 클릭
2. "새 보안 비밀" → 이름: `OPENAI_API_KEY`
3. 값에 본인 키 붙여넣고 **노트북 액세스 ON**

> ⚠️ 키를 셀에 직접 쓰지 마세요. GitHub에 올리면 10분 안에 탈취됩니다.

In [ ]:
from google.colab import userdata
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

# 키 점검 (앞 7자리만 찍어보기)
print(f"Key prefix: {OPENAI_API_KEY[:7]}... (length: {len(OPENAI_API_KEY)})")

client = OpenAI(api_key=OPENAI_API_KEY)

## Step 1. 테마 선택 & 데이터 수집

### 🎯 택 1

| 테마 | 종목 | 티커 |
|---|---|---|
| 🔥 반도체 | 삼성전자 / SK하이닉스 / 한미반도체 | 005930 / 000660 / 042700 |
| 🔋 2차전지 | LG에너지솔루션 / 에코프로 / POSCO홀딩스 | 373220 / 086520 / 005490 |
| 🧬 바이오 | 셀트리온 / 삼성바이오로직스 / 유한양행 | 068270 / 207940 / 000100 |

In [ ]:
# TODO: 본인이 고른 테마로 수정
target_stocks = {
    "삼성전자": "005930",
    "SK하이닉스": "000660",
    "한미반도체": "042700",
}

# 오늘 기준 1년치
end = datetime.today().strftime("%Y%m%d")
start = (datetime.today() - timedelta(days=365)).strftime("%Y%m%d")
print(f"조회 기간: {start} ~ {end}")

In [ ]:
# 종목별 OHLCV 조회
dfs = {}
for name, ticker in target_stocks.items():
    df = stock.get_market_ohlcv_by_date(start, end, ticker)
    dfs[name] = df
    print(f"✅ {name} ({ticker}): {len(df)}일치 수집")

# 첫 종목 데이터 미리보기
first_name = list(dfs.keys())[0]
dfs[first_name].tail()

## Step 2. 이동평균선 + Plotly 캔들차트

### 💡 왜 이동평균선을 볼까?

- **MA5** (5일): 단기 추세 — "이번 주 분위기"
- **MA20** (20일): 중기 추세 — "한 달 흐름"
- **MA60** (60일): 장기 추세 — "분기 흐름"

> MA5가 MA20을 위로 뚫으면 **골든크로스**, 아래로 뚫으면 **데드크로스**.

In [ ]:
# 모든 종목에 MA 컬럼 추가
for name, df in dfs.items():
    df['MA5']  = df['종가'].rolling(window=5).mean()
    df['MA20'] = df['종가'].rolling(window=20).mean()
    df['MA60'] = df['종가'].rolling(window=60).mean()

dfs[first_name][['종가', 'MA5', 'MA20', 'MA60']].tail()

In [ ]:
def make_chart(name: str, df: pd.DataFrame) -> go.Figure:
    """한 종목의 캔들차트 + 이평선 3개를 그린다."""
    fig = go.Figure(data=[
        go.Candlestick(
            x=df.index,
            open=df['시가'], high=df['고가'],
            low=df['저가'], close=df['종가'],
            name='캔들',
            increasing_line_color='#EF4444',  # 상승 빨강 (한국 관례)
            decreasing_line_color='#3B82F6',  # 하락 파랑
        ),
        go.Scatter(x=df.index, y=df['MA5'],  line=dict(color='orange',    width=1.3), name='MA5'),
        go.Scatter(x=df.index, y=df['MA20'], line=dict(color='royalblue', width=1.3), name='MA20'),
        go.Scatter(x=df.index, y=df['MA60'], line=dict(color='firebrick', width=1.3), name='MA60'),
    ])
    fig.update_layout(
        title=f'{name} — 1년 주가 & 이동평균선',
        template='plotly_dark',
        xaxis_rangeslider_visible=False,
        height=600,
        legend=dict(orientation='h', y=1.05),
    )
    return fig

# 첫 종목 시각화
fig = make_chart(first_name, dfs[first_name])
fig.show()

## Step 3. 🌟 AI Vision 분석

오늘의 하이라이트. 차트 이미지를 AI에게 보여주고 해석을 받는다.

In [ ]:
def encode_image(path: str) -> str:
    """이미지 파일을 base64 문자열로 인코딩."""
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

In [ ]:
SYSTEM_PROMPT = """
너는 월스트리트 15년차 헤지펀드 퀀트 애널리스트야.
첨부된 주가 차트의 이동평균선(주황=MA5, 파랑=MA20, 빨강=MA60) 배열과
캔들 패턴을 분석해. 금융 전문 용어를 쓰되 초보자도 이해할 수 있게 설명해.

반드시 아래 형식으로만 답해:

📊 현재 추세: (한 줄 요약)
🎯 핵심 시그널: (골든크로스/데드크로스/정배열/역배열 등 구체적 패턴 1~2개)
💡 투자 의견: [매수 / 보유 / 매도] 중 하나 + 3줄 이내 근거
⚠️ 리스크: (놓치면 안 될 하방 리스크 한 줄)
"""

def analyze_with_ai(name: str, image_path: str) -> str:
    """GPT-4o Vision에게 차트 이미지를 보내서 분석 텍스트를 받는다."""
    image_b64 = encode_image(image_path)
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": [
                {"type": "text",
                 "text": f"이 차트는 {name}의 최근 1년 일봉이야. 이평선 배열을 중심으로 분석해줘."},
                {"type": "image_url",
                 "image_url": {"url": f"data:image/png;base64,{image_b64}"}}
            ]}
        ],
        max_tokens=500,
        temperature=0.3,
    )
    return response.choices[0].message.content

In [ ]:
# 첫 종목으로 테스트
fig.write_image("chart.png", width=1200, height=700, scale=2)
analysis = analyze_with_ai(first_name, "chart.png")
print(f"🤖 AI 퀀트 의견 — {first_name}\n{'='*40}")
print(analysis)

### 🔁 모든 종목 반복

아래 셀은 모든 선택 종목에 대해 차트 생성 → AI 분석을 자동 실행한다.
종목당 약 5~10초 걸림.

In [ ]:
results = {}

for name, df in dfs.items():
    print(f"\n⏳ {name} 분석 중...")
    fig = make_chart(name, df)
    img_path = f"chart_{name}.png"
    fig.write_image(img_path, width=1200, height=700, scale=2)
    analysis = analyze_with_ai(name, img_path)
    results[name] = analysis
    print(f"✅ {name} 완료")

# 결과 출력
for name, text in results.items():
    print(f"\n{'='*50}\n🤖 {name}\n{'='*50}\n{text}")

## Step 4. 결과 저장 (2주차 재료)

다음 주 포트폴리오 비중 계산에서 이 CSV를 그대로 읽어 쓴다.

In [ ]:
summary = pd.DataFrame([
    {
        "종목": name,
        "티커": target_stocks[name],
        "현재가": int(dfs[name]['종가'].iloc[-1]),
        "MA20": round(dfs[name]['MA20'].iloc[-1], 0),
        "MA60": round(dfs[name]['MA60'].iloc[-1], 0),
        "1년_수익률_%": round(
            (dfs[name]['종가'].iloc[-1] / dfs[name]['종가'].iloc[0] - 1) * 100, 2
        ),
        "AI_의견": results[name],
    }
    for name in dfs
])

filename = f"analysis_{end}.csv"
summary.to_csv(filename, index=False, encoding='utf-8-sig')
print(f"💾 저장 완료: {filename}")
summary

---

## 📝 회고 (과제)

아래 세 질문에 본인의 답을 적어보세요. (이 셀을 더블클릭 → 편집)

**Q1. AI 분석에서 가장 납득 간 부분과 납득 안 간 부분은?**

> (여기에 답)

**Q2. 3종목 중 실제로 100만원 넣는다면 어디에 넣고, 왜?**

> (여기에 답)

**Q3. 차트 외에 AI가 뭘 더 봤으면 분석이 정확해졌을까?**

> (여기에 답)

---

### 🏁 완료 체크리스트

- [ ] 종목 3개 모두 차트 생성
- [ ] AI 분석 3건 모두 수신
- [ ] `analysis_YYYYMMDD.csv` 저장
- [ ] 회고 3문항 작성
- [ ] API 키가 코드에 하드코딩되지 않음 ✅

### 🔜 다음 주 예고

> 오늘 AI가 종목별로 '매수/보유/매도' 의견을 내놨죠?
> 다음 주에는 이걸 근거로 **포트폴리오 비중**을 짜고,
> AI에게 "이 비중 구성 이상한 거 없어?" 라고 시비를 걸어봅니다.